# **SNU AI Challenge Baseline Code**

이 코드는 SNU AI Challenge 경진대회의 베이스라인 코드입니다.

## **Task**

주어진 스토리라인(캡션)에 맞게 4개의 이미지 프레임을 올바른 순서로 재배열하는 문제를 해결해야 합니다.

- **Input**: 자연어 문장과 여러 장의 프레임으로 구성된 데이터
  - 예: `{ "text": "자연어 문장", "frames": [image_3, image_1, image_4, image_2] }`
- **Output**: 정답 순서대로 다시 배열하였을 때 각 프레임의 위치
  - 예: `[3, 4, 1, 2]`
  - 정답 순서대로 다시 배열하였을 때 첫 번째 프레임은 3번째에 위치, 두 번째 프레임은 4번째에 위치, …

## **Baseline**

`Qwen2-VL-2B-Instruct` 모델을 사용하여 별도의 학습 없이 프롬프트만으로 추론을 진행하는 코드입니다.

## **Overall Pipeline**

| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 로드 (test.csv + 이미지) |
| Step 2 | 모델 로드 (Qwen2-VL-2B-Instruct from HuggingFace) |
| Step 3 | 추론 진행 (프롬프트 → 모델 생성 → 출력 파싱) |
| Step 4 | 결과 저장 (submission.csv) |


## 0. 환경 설정 및 패키지 설치


In [ ]:
%pip install -q torch torchvision transformers pandas pillow tqdm accelerate qwen-vl-utils

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import ast
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

/opt/homebrew/anaconda3/envs/snuaichallenge/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 라이브러리 임포트 및 경로 설정

아래 셀에서 **`OUTPUT_DIR` 와 `DATA_DIR`** 경로를 본인의 환경에 맞게 수정하세요.


In [ ]:
# --- 설정 (본인 환경에 맞게 수정) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

OUTPUT_DIR = "./outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_DIR = './snuaichallenge_data/'
TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
TEST_IMAGE_DIR = os.path.join(DATA_DIR, 'test')

## 2. 데이터 로드 & 모델 로드

In [ ]:
# --- 데이터 로드 ---
test_df = pd.read_csv(TEST_CSV)
print(f"Test samples: {len(test_df)}")

# --- 모델 및 프로세서 로드 ---
print(f"Loading model: {MODEL_NAME}...")
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

Test samples: 819
Loading model: Qwen/Qwen2-VL-2B-Instruct...


In [ ]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Model loaded successfully.")

Loading weights: 100%|██████████| 729/729 [00:07<00:00, 103.80it/s]
Some parameters are on the meta device because they were offloaded to the disk.


Model loaded successfully.


## 3. 추론 (Inference)


In [ ]:
# --- 추론 함수 정의 ---
def get_prompt_message(row, image_dir):
    """
    4장의 프레임과 문장을 조합하여 Qwen2-VL-2B-Instruct 모델에 보낼 프롬프트 메시지를 구성합니다.
    """
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, row['Id'], img_file)
        content.append({
            "type": "image",
            "image": img_path,
        })
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]
    return messages

In [ ]:
def parse_model_output(output_text):
    """
    모델의 텍스트 출력에서 리스트 부분만 추출하고, 대회 정답 형식에 맞게 변환합니다.
    예시: 모델이 "The answer is [4, 2, 1, 3]" 이라 출력 -> [3, 2, 4, 1] 반환
    """
    try:
        start_idx = output_text.find('[')
        end_idx = output_text.rfind(']')

        if start_idx != -1 and end_idx != -1:
            list_str = output_text[start_idx : end_idx+1]
            result = ast.literal_eval(list_str)

            if isinstance(result, list) and len(result) == 4 and sorted(result) == [1, 2, 3, 4]:
                submission_answer = [0] * 4
                for index, image_num in enumerate(result):
                    submission_answer[image_num - 1] = index + 1

                return submission_answer

    except:
        pass

    return [1, 2, 3, 4]

In [ ]:
# --- 추론 실행 ---
predictions = []

print("Starting inference...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    messages = get_prompt_message(row, TEST_IMAGE_DIR)

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    pred_list = parse_model_output(output_text)

    predictions.append({
        'Id': row['Id'],
        'Answer': str(pred_list)
    })

Starting inference...


  4%|▍         | 31/819 [1:17:05<33:20:06, 152.29s/it]

## 4. 결과 저장 (Submission)

### 제출 파일 형식 (submission.csv)

제출 파일은 반드시 아래 형식을 따라야 합니다:

| 컬럼 | 타입 | 설명 | 예시 |
|------|------|------|------|
| `Id` | string | 테스트 샘플 고유 ID | `WI8o3F` |
| `Answer` | string | 정답 순서대로 다시 배열하였을 때 각 프레임의 위치 (Python 리스트 형태의 문자열) | `[3, 1, 4, 2]` |

#### 올바른 예시

```csv
Id,Answer
WI8o3F,[3, 1, 4, 2]
Ae7zLm,[1, 2, 3, 4]
...


In [ ]:
# --- 제출 파일 생성 ---
submit_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission_df = pd.DataFrame(predictions)
submission_df.to_csv(submit_path, index=False)

print(f"Inference completed. Saved to {submit_path}")
print(submission_df.head())

Inference completed. Saved to ./outputs/baseline_v3/submission.csv
       Id        Answer
0  hIgt7V  [1, 2, 4, 3]
1  9TGazx  [1, 2, 3, 4]
2  e0H1Ul  [1, 2, 3, 4]
3  gKAo5J  [3, 1, 4, 2]
4  U2JY4q  [1, 2, 3, 4]
